# 阶段七：串与压缩存储 —— 从零到手写 KMP 的完全教程

---

## 第一部分：串（String）

---

## 4.1.1 串的定义和基本操作

### 一、什么是"串"？

**串（String）** 是由零个或多个字符组成的**有限序列**。

数学记法：`S = 'a₁a₂a₃…aₙ'`

- `S` 是**串名**
- 单引号括起来的部分是**串值**
- `aᵢ` 是串中的单个字符
- `n` 是**串的长度**（字符个数）
- 当 `n = 0` 时，称为**空串（Empty String）**，记作 `Ø` 或 `""`

> ⚠️ **空串 ≠ 空格串**：`""` 长度为 0；`"   "` 长度为 3，里面有 3 个空格字符。

### 二、几个重要术语

| 术语 | 含义 | 示例（S = `"abcde"`） |
|------|------|----------------------|
| **子串** | 串中任意个**连续字符**组成的子序列 | `"bcd"`、`"a"`、`""` |
| **主串** | 包含子串的那个串 | `"abcde"` 是 `"bcd"` 的主串 |
| **字符位置** | 字符在串中的序号（**从 1 开始**） | `'a'` 的位置是 1，`'c'` 的位置是 3 |
| **子串位置** | 子串第一个字符在主串中的位置 | `"bcd"` 在主串中的位置是 2 |
| **串相等** | 两个串长度相同且对应位置字符都相同 | `"abc"` == `"abc"` ✅ |

> 📌 **注意**：数据结构教材中，串的下标通常**从 1 开始**（区别于 C/C++ 数组从 0 开始）。在后续代码实现中，我们会特别处理这个差异。

### 三、串的基本操作

串作为一种**抽象数据类型 (ADT)**，需要定义以下基本操作：

```
StrAssign(&T, chars)      // 赋值：把串 chars 赋给 T
StrCopy(&T, S)             // 复制：把串 S 复制给 T
StrEmpty(S)                // 判空：S 是否为空串
StrLength(S)               // 求长度：返回 S 的元素个数
ClearString(&S)            // 清空：将 S 清为空串
DestroyString(&S)          // 销毁：释放 S 的存储空间
Concat(&T, S1, S2)         // 连接：用 T 返回 S1 和 S2 连接而成的新串
SubString(&Sub, S, pos, len) // 求子串：用 Sub 返回 S 中第 pos 个字符起长度为 len 的子串
Index(S, T, pos)           // 定位：返回子串 T 在主串 S 中从第 pos 个字符起首次出现的位置
StrCompare(S, T)           // 比较：若 S > T 返回正数，S == T 返回 0，S < T 返回负数
```

#### 核心操作详解

**1. `SubString(&Sub, S, pos, len)` —— 求子串**

这是最基础也最重要的操作之一。

```
主串 S = "Hello World"
SubString(Sub, S, 7, 5)
结果：Sub = "World"
解释：从第 7 个字符开始，取 5 个字符
```

约束条件：`1 ≤ pos ≤ StrLength(S)` 且 `0 ≤ len ≤ StrLength(S) - pos + 1`

**2. `StrCompare(S, T)` —— 串的比较**

串的比较采用**字典序（Lexicographic Order）**，逐字符比较 ASCII 码：

```
StrCompare("abc", "abd")  → 返回负数（因为 'c' < 'd'）
StrCompare("abc", "ab")   → 返回正数（前缀相同但 S 更长）
StrCompare("abc", "abc")  → 返回 0
```

比较规则：从左到右逐个比较，遇到第一个不同的字符就比较它们的 ASCII 码；如果所有字符相同，则比较长度。

**3. `Index(S, T, pos)` —— 定位（模式匹配）**

在主串 S 中，从第 pos 个位置开始，查找子串 T 第一次出现的位置。如果找不到返回 0。

```
S = "ababcabcac"
T = "abc"
Index(S, T, 1) → 返回 3（从位置 3 开始匹配上 "abc"）
```

> 💡 **重要认识**：串的所有复杂操作，理论上都可以用 `SubString`、`StrCompare`、`Concat` 等基本操作组合而成。但为了效率，我们通常会设计专门的算法（如 KMP）来实现模式匹配。

---

## 4.1.2 串的存储结构

串的存储结构主要有三种：**定长顺序存储**、**堆分配存储**、**块链存储**。

### 一、定长顺序存储（静态数组）

用一个**固定大小的字符数组**来存储串。

In [ ]:
#define MAXLEN 255  // 预定义最大串长

In [ ]:
typedef struct {
    char ch[MAXLEN + 1];  // 存储串的字符数组，ch[0] 不用
    int length;       // 串的实际长度
} SString;

#### 存储方案的选择

教材中常见两种处理下标的方式：

**方案一：`ch[0]` 不用，从 `ch[1]` 开始存放**（与教材公式一致）

```
下标:  0    1    2    3    4    5
内容: [×]  [H]  [e]  [l]  [l]  [o]
       ↑    ↑                   ↑
     不用  第1个字符          第5个字符
length = 5
```

**方案二：从 `ch[0]` 开始存放**（与 C 语言习惯一致）

```
下标:  0    1    2    3    4
内容: [H]  [e]  [l]  [l]  [o]
       ↑                   ↑
     第1个字符          第5个字符
length = 5
```

> 📌 **本教程采用方案一**（`ch[0]` 不用），与王道教材和考研真题保持一致。这样做的好处是：子串位置、字符位置都从 1 开始，与理论公式完全对应。

#### 基本操作实现

In [ ]:
// 求子串：用 Sub 返回 S 中第 pos 个字符起长度为 len 的子串

In [ ]:
bool SubString(SString &Sub, SString S, int pos, int len) {
    // 越界判断
    if (pos < 1 || len < 0 || pos + len - 1 > S.length)
        return false;

    for (int i = pos; i < pos + len; i++) {
        Sub.ch[i - pos + 1] = S.ch[i];
    }
    Sub.length = len;
    return true;
}

// 比较操作：S > T 返回正数，S == T 返回 0，S < T 返回负数

In [ ]:
int StrCompare(SString S, SString T) {
    for (int i = 1; i <= S.length && i <= T.length; i++) {
        if (S.ch[i] != T.ch[i])
            return S.ch[i] - T.ch[i];
    }
    // 前面都相同，比较长度
    return S.length - T.length;
}

#### 优缺点

| 优点 | 缺点 |
|------|------|
| 实现简单，随机访问 O(1) | 长度固定；超过 MAXLEN 时必须由操作显式拒绝、报错或按约定截断，否则会越界 |
| 无需动态内存管理 | 浪费空间（若串很短） |

### 二、堆分配存储（动态数组）

用 `malloc` / `new` 在**堆区**动态分配一片连续空间。

In [ ]:
typedef struct {
    char *ch;     // 指向动态分配的字符数组
    int length;   // 串的实际长度
} HString;

In [ ]:
// 初始化
bool InitString(HString &S, const char *str) {
    S.length = strlen(str);
    S.ch = (char *)malloc(sizeof(char) * (S.length + 1));
    // +1 是因为 ch[0] 不用
    if (S.ch == NULL) return false;
    for (int i = 0; i < S.length; i++) {
        S.ch[i + 1] = str[i];  // 从下标 1 开始存
    }
    return true;
}

#### 优缺点

| 优点 | 缺点 |
|------|------|
| 长度灵活，按需分配 | 需要手动管理内存（malloc/free） |
| 本质仍是顺序存储，支持随机访问 | 扩容需要 realloc，可能涉及数据搬移 |

> 💡 `std::string` 是管理连续字符序列的 RAII 封装；实现可使用堆分配，也可使用小字符串优化（SSO），C++ 标准不规定其具体存储策略。

### 三、块链存储（链式存储）

用**链表**存储串，每个结点存放若干个字符。

In [ ]:
#define BLOCK_SIZE 4  // 每个块存 4 个字符

In [ ]:
typedef struct StringNode {
    char ch[BLOCK_SIZE]; // 每个结点存多个字符
    struct StringNode *next;
} StringNode, *BlockString;

示例：存储 `"Hello World!!"` （13 个字符）

```
[H][e][l][l] → [o][ ][W][o] → [r][l][d][!] → [!][#][#][#]
  块1(4字符)     块2(4字符)     块3(4字符)     块4(1有效+3填充)
```

最后一个块不满时，用特殊字符（如 `#` 或 `\0`）填充。

#### 存储密度

$$\text{存储密度} = \frac{\text{串值所占的存储位}}{\text{实际分配的存储位}}$$

- 块越大 → 存储密度越高 → 但插入删除越不便
- 块越小（极端：每块 1 个字符）→ 存储密度越低 → 退化为普通链表

#### 优缺点

| 优点 | 缺点 |
|------|------|
| 插入/删除方便 | 存储密度低，空间利用率差 |
| 长度无限制 | 不支持随机访问 |
| | 实现复杂，实际工程中很少使用 |

### 四、三种存储结构对比总结

| 特性 | 定长顺序 | 堆分配 | 块链 |
|------|---------|--------|------|
| 空间分配 | 编译时静态分配 | 运行时动态分配 | 运行时动态分配 |
| 随机访问 | ✅ O(1) | ✅ O(1) | ❌ O(n) |
| 长度限制 | 有（MAXLEN） | 无 | 无 |
| 插入删除 | 需移动元素 | 需移动元素 | 较方便 |
| 实际应用 | 简单场景 | **最常用** | 很少用 |

---

## 第二部分：模式匹配算法

> **模式匹配**：在**主串 S** 中寻找**模式串 T** 出现的位置。这是串最核心的操作。

---

## 4.2.1 朴素模式匹配算法（Brute-Force）

### 一、核心思想

从主串的第一个字符开始，依次与模式串的字符逐一比较：
- 如果当前字符匹配，继续比较下一个字符
- 如果某个字符不匹配，**主串回退**到本次匹配起点的下一个位置，模式串回到开头，重新开始匹配

### 二、图解过程

```
主串 S = "ababcabcac"  (长度 10)
模式串 T = "abcac"      (长度 5)

第 1 趟匹配：
  S: a b a b c a b c a c
  T: a b c
        ↑ 失配！S[3]='a' ≠ T[3]='c'
  → 主串指针 i 回退到 2，模式串指针 j 回退到 1

第 2 趟匹配：
  S: a b a b c a b c a c
     . b
       ↑ 失配！S[2]='b' ≠ T[1]='a'
  → i 回退到 3，j 回退到 1

第 3 趟匹配：
  S: a b a b c a b c a c
      . . a b c a c
  S[3]='a'=T[1]='a' ✓
  S[4]='b'=T[2]='b' ✓
  S[5]='c'=T[3]='c' ✓
  S[6]='a'=T[4]='a' ✓
  S[7]='b'≠T[5]='c' ✗ 失配！
  → i 回退到 4，j 回退到 1

第 4 趟匹配：
  S[4]='b'≠T[1]='a' ✗ 立即失配
  → i 回退到 5，j 回退到 1

第 5 趟匹配：
  S[5]='c'≠T[1]='a' ✗ 立即失配
  → i 回退到 6，j 回退到 1

第 6 趟匹配：
  S[6]='a'=T[1]='a' ✓
  S[7]='b'=T[2]='b' ✓
  S[8]='c'=T[3]='c' ✓
  S[9]='a'=T[4]='a' ✓
  S[10]='c'=T[5]='c' ✓
  → 全部匹配！返回位置 6
```

### 三、代码实现

In [ ]:
// 朴素模式匹配算法
// 在主串 S 中查找模式串 T，返回第一次出现的位置（从1开始），找不到返回 0

In [ ]:
int Index_BF(SString S, SString T) {
    int i = 1;  // 主串指针，从第 1 个字符开始
    int j = 1;  // 模式串指针，从第 1 个字符开始

    while (i <= S.length && j <= T.length) {
        if (S.ch[i] == T.ch[j]) {
            // 当前字符匹配，继续比较下一个
            i++;
            j++;
        } else {
            // 不匹配：主串回退，模式串归位
            i = i - j + 2;  // ★ 关键公式：回退到上次起点的下一个位置
            j = 1;           // 模式串从头开始
        }
    }

    if (j > T.length) {
        // 模式串已全部匹配
        return i - T.length;  // 返回匹配起始位置
    } else {
        return 0;  // 匹配失败
    }
}

### 四、关键公式推导：`i = i - j + 2`

为什么失配时 `i` 要回退到 `i - j + 2`？

```
匹配开始时：i 指向主串某位置 start，j = 1
匹配过程中：i 和 j 同步前进了 (j-1) 步
所以此时：start = i - (j - 1) = i - j + 1
下一趟应从 start + 1 = i - j + 2 开始
```

这就是 `i = i - j + 2` 的由来。

### 五、复杂度分析

设主串长度为 n，模式串长度为 m。

| 情况 | 时间复杂度 | 说明 |
|------|-----------|------|
| 最好情况 | O(m) | 第一趟就匹配成功 |
| 最坏情况 | **O(n·m)** | 每趟都比较到模式串最后一个字符才失配 |
| 平均情况（随机文本） | O(n+m) | 实际中大多数情况效率还可以 |

**最坏情况示例**：

```
S = "aaaa...aaab"  （n-1 个 a，1 个 b）
T = "aaa...ab"     （m-1 个 a，1 个 b）

每一趟都要比较 m 次才发现最后一个字符不匹配
总共需要 (n-m+1) 趟
总比较次数 ≈ (n-m+1) × m ≈ O(n·m)
```

> ❗ 朴素算法的主要缺陷：**主串指针 i 会回退**。在失配时，i 退回到本次匹配起点的下一位，之前的比较信息全部浪费。KMP 算法正是为了解决这个问题而诞生的。

---

## 4.2.2_1 KMP 算法

### 一、KMP 的核心思想

**KMP 算法的精髓**：当发生失配时，**主串指针 i 不回退**，只移动模式串指针 j。

这个想法的关键在于：当 `S[i] ≠ T[j]` 时，我们已经知道 `T[1..j-1]` 与 `S[i-j+1..i-1]` 匹配成功了。这些**已匹配的信息**不应该被浪费！

### 二、直觉理解

假设匹配到如下状态时失配了：

```
主串 S: ... a b a b a b a ? ...
                         ↑ i (失配位置)
模式串T:     a b a b a c
                       ↑ j=6 (T[6]='c'，失配)
```

此时已知：`S[i-5..i-1] = T[1..5] = "ababa"`

朴素算法会把 i 退回，从头比较。但仔细观察已匹配的 `"ababa"`：

```
T[1..5] = "a b a b a"
           ─────       前缀 "aba"
               ─────   后缀 "aba"
```

`"ababa"` 有一个长度为 3 的**既是前缀又是后缀**的子串 `"aba"`！

这意味着：已匹配部分的**末尾 3 个字符** `= S[i-3..i-1] = "aba"` 恰好等于模式串的**开头 3 个字符** `T[1..3] = "aba"`。

所以我们不需要从头比较，可以直接让 `j` 跳到 4，继续用 `T[4]` 和 `S[i]` 比较：

```
主串 S: ... a b a b a b a ? ...
                         ↑ i 不动！
模式串T:         a b a b a c
                         ↑ j 跳到 4
```

### 三、next 数组的含义

为了知道失配时 j 应该跳到哪里，我们预计算一个 **`next` 数组**：

> **`next[j]`**：当模式串第 j 个字符与主串失配时，j 应该回退到的位置。

更严格的定义：

$$\text{next}[j] = \begin{cases} 0, & j = 1 \text{（第 1 个字符就失配，需要 i 右移一位）} \\[6pt] \max\{k \mid 1 \leq k < j \text{ 且 } T[1..k-1] = T[j-k+1..j-1]\}, & j > 1 \end{cases}$$

用人话说：**`next[j]` 等于模式串 `T[1..j-1]`（即 j 前面那些已匹配的字符）的最长相等真前后缀长度 + 1**。

### 四、手算 next 数组示例

以 `T = "abcabd"` 为例（长度 6）：

```
j = 1: next[1] = 0     （固定值）

j = 2: 考察 T[1..1] = "a"
       真前缀集合 = {}
       真后缀集合 = {}
       最长公共 = 0 → next[2] = 0 + 1 = 1

j = 3: 考察 T[1..2] = "ab"
       真前缀集合 = {"a"}
       真后缀集合 = {"b"}
       没有相等的 → next[3] = 0 + 1 = 1

j = 4: 考察 T[1..3] = "abc"
       真前缀集合 = {"a", "ab"}
       真后缀集合 = {"c", "bc"}
       没有相等的 → next[4] = 0 + 1 = 1

j = 5: 考察 T[1..4] = "abca"
       真前缀集合 = {"a", "ab", "abc"}
       真后缀集合 = {"a", "ca", "bca"}
       最长公共 = "a"（长度 1）→ next[5] = 1 + 1 = 2

j = 6: 考察 T[1..5] = "abcab"
       真前缀集合 = {"a", "ab", "abc", "abca"}
       真后缀集合 = {"b", "ab", "cab", "bcab"}
       最长公共 = "ab"（长度 2）→ next[6] = 2 + 1 = 3
```

结果：

| j | 1 | 2 | 3 | 4 | 5 | 6 |
|---|---|---|---|---|---|---|
| T[j] | a | b | c | a | b | d |
| next[j] | 0 | 1 | 1 | 1 | 2 | 3 |

### 五、KMP 匹配算法代码

In [ ]:
// KMP 模式匹配算法
// 前提：next[] 数组已经计算好

In [ ]:
int Index_KMP(SString S, SString T, int next[]) {
    int i = 1;  // 主串指针
    int j = 1;  // 模式串指针

    while (i <= S.length && j <= T.length) {
        if (j == 0 || S.ch[i] == T.ch[j]) {
            // j==0 表示模式串第 1 个字符就失配了，此时 i 和 j 各前进一步
            // 字符匹配时也是 i 和 j 各前进一步
            i++;
            j++;
        } else {
            // 失配：i 不动，j 回退到 next[j]
            j = next[j];  // ★ 核心：主串不回退！
        }
    }

    if (j > T.length)
        return i - T.length;  // 匹配成功
    else
        return 0;             // 匹配失败
}

#### 关键点解析

**1. 为什么要判 `j == 0`？**

当 `next[j] = 0` 时（即 j 已经退到 0），说明模式串的第一个字符就和主串当前字符不匹配。此时需要：
- `i++`：主串前进一位
- `j++`：j 从 0 变回 1，下一轮从模式串头开始

把 `j == 0` 和匹配成功合并到一个 if 分支，非常巧妙。

**2. 对比朴素算法**

```
朴素算法：失配时 i = i-j+2, j = 1    ← i 回退了！
KMP算法： 失配时 j = next[j]          ← i 不动！只移动 j
```

### 六、KMP 匹配完整图解

```
S = "ababcabcacbab"
T = "abcac"
next = {0, 1, 1, 1, 2}

第 1 趟：
  S: a b a b c a b c a c b a b
  T: a b c
     ✓ ✓ ✗  S[3]='a' ≠ T[3]='c'
  j=3, next[3]=1 → j 跳到 1
  i 保持在 3 不动

第 2 趟（i=3, j=1）：
  S: a b a b c a b c a c b a b
         a b c a c
         ✓ ✓ ✓ ✓ ✗  S[7]='b' ≠ T[5]='c'
  j=5, next[5]=2 → j 跳到 2
  i 保持在 7 不动

第 3 趟（i=7, j=2）：
  S: a b a b c a b c a c b a b
               b c a c
  S[7]='b', T[2]='b' → ✓
  S[8]='c', T[3]='c' → ✓
  S[9]='a', T[4]='a' → ✓
  S[10]='c', T[5]='c' → ✓
  j > T.length → 匹配成功！
  返回 i - T.length = 11 - 5 = 6
```

### 七、KMP 的时间复杂度

| 阶段 | 复杂度 |
|------|--------|
| 预处理 next 数组 | O(m) |
| 匹配过程 | O(n) |
| **总计** | **O(n + m)** |

对比朴素算法的 O(n·m)，KMP 有质的提升！

---

## 4.2.2_2 求 next 数组

### 一、手算方法回顾

我们已经知道：`next[j]` = `T[1..j-1]` 的最长相等真前后缀长度 + 1。

但如何用**程序**高效计算 next 数组？

### 二、递推思路（精华）

这是 KMP 中最难理解但最精妙的部分。

假设已经知道 `next[j] = k`，即 `T[1..k-1] = T[j-k..j-1]`（最长相等真前后缀长度为 k-1）。

现在要求 `next[j+1]`，需要考察 `T[1..j]` 的最长相等真前后缀。

**情况 1：`T[k] == T[j]`**

```
T[1..j]:    T₁ T₂ ... Tₖ₋₁ [Tₖ] ... Tⱼ₋ₖ₊₁ ... Tⱼ₋₁ [Tⱼ]
            ↑ 前缀 k-1 位 ↑              ↑ 后缀 k-1 位 ↑
            └─── 已知相等 ──┘              └─── 已知相等 ──┘
                         [Tₖ] == [Tⱼ]  ← 新增的也相等！
```

那么 `T[1..k] = T[j-k+1..j]`，最长相等真前后缀长度变为 k。

所以 `next[j+1] = k + 1 = next[j] + 1`

**情况 2：`T[k] ≠ T[j]`**

此时不能简单地把前后缀延长。我们需要在 `T[1..k-1]` 中找一个更短的前后缀来尝试。

关键洞察：**这个过程本身就是一个"模式匹配"问题！**

我们让 `k = next[k]`，回退到更短的前后缀，再检查 `T[k'] == T[j]`？

如果还不等，继续回退 `k = next[k]`，直到 `k == 0`，说明不存在任何相等的真前后缀，此时 `next[j+1] = 1`。

### 三、代码实现

In [ ]:
// 求模式串 T 的 next 数组

In [ ]:
void GetNext(SString T, int next[]) {
    int j = 1;    // 当前位置
    int k = 0;    // next[j] 的候选值
    next[1] = 0;  // 固定值

    while (j < T.length) {
        if (k == 0 || T.ch[j] == T.ch[k]) {
            // 情况 1：T[j] == T[k]，或 k 已退到 0
            j++;
            k++;
            next[j] = k;
        } else {
            // 情况 2：T[j] ≠ T[k]，k 回退
            k = next[k];
        }
    }
}

> 🔍 **你发现了吗？** 这段代码的结构和 KMP 匹配算法**几乎一模一样**！因为求 next 数组本质上就是模式串在自己身上做匹配。

### 四、完整推演示例

以 `T = "ababaaab"` 为例，详细推演整个过程：

```
初始：j=1, k=0, next[1]=0

─── 第 1 轮 ───
k==0 → 进入 if 分支
j++→j=2, k++→k=1
next[2] = 1

─── 第 2 轮 ───
j=2, k=1
T[2]='b' vs T[1]='a' → 不等
k = next[1] = 0

─── 第 3 轮 ───
k==0 → 进入 if 分支
j++→j=3, k++→k=1
next[3] = 1

─── 第 4 轮 ───
j=3, k=1
T[3]='a' vs T[1]='a' → 相等！
j++→j=4, k++→k=2
next[4] = 2

─── 第 5 轮 ───
j=4, k=2
T[4]='b' vs T[2]='b' → 相等！
j++→j=5, k++→k=3
next[5] = 3

─── 第 6 轮 ───
j=5, k=3
T[5]='a' vs T[3]='a' → 相等！
j++→j=6, k++→k=4
next[6] = 4

─── 第 7 轮 ───
j=6, k=4
T[6]='a' vs T[4]='b' → 不等
k = next[4] = 2

─── 第 8 轮 ───
j=6, k=2
T[6]='a' vs T[2]='b' → 不等
k = next[2] = 1

─── 第 9 轮 ───
j=6, k=1
T[6]='a' vs T[1]='a' → 相等！
j++→j=7, k++→k=2
next[7] = 2

─── 第 10 轮 ───
j=7, k=2
T[7]='a' vs T[2]='b' → 不等
k = next[2] = 1

─── 第 11 轮 ───
j=7, k=1
T[7]='a' vs T[1]='a' → 相等！
j++→j=8, k++→k=2
next[8] = 2

j=8 == T.length → 循环结束
```

最终结果：

| j | 1 | 2 | 3 | 4 | 5 | 6 | 7 | 8 |
|---|---|---|---|---|---|---|---|---|
| T[j] | a | b | a | b | a | a | a | b |
| next[j] | 0 | 1 | 1 | 2 | 3 | 4 | 2 | 2 |

### 五、next 数组的验证方法

拿到 next 数组后，可以逐个验证：

```
next[6] = 4 → T[1..5]="ababa" 的最长相等真前后缀长度 = 4-1 = 3
验证：T[1..3]="aba" 和 T[3..5]="aba" → 确实相等 ✓
长度为 3 确实是最长 ✓
```

---

## 4.2.3 KMP 算法的进一步优化 —— nextval 数组

### 一、next 数组的缺陷

考虑这个例子：

```
S = "aaabaaaab"
T = "aaaab"
next = {0, 1, 2, 3, 4}
```

匹配过程：

```
S: a a a b a a a a b
T: a a a a b
         ↑ S[4]='b' ≠ T[4]='a'，失配

j=4, next[4]=3 → 跳到 j=3
T[3]='a'，但 S[4]='b'，还是不等！（因为 T[3]==T[4]）

j=3, next[3]=2 → 跳到 j=2
T[2]='a'，但 S[4]='b'，还是不等！（因为 T[2]==T[3]==T[4]）

j=2, next[2]=1 → 跳到 j=1
T[1]='a'，但 S[4]='b'，还是不等！（因为 T[1]==T[2]==T[3]==T[4]）

j=1, next[1]=0 → j=0
```

**问题**：当 `T[j] == T[next[j]]` 时，跳过去之后一定还是失配，白白浪费了比较次数！

### 二、优化思路

当 `T[j] == T[next[j]]` 时，说明跳到 `next[j]` 之后面对的字符和当前一样，仍然会失配。应该继续往前跳，直到遇到一个不同的字符。

这就是 **nextval 数组**的思想。

### 三、nextval 的定义

$$\text{nextval}[j] = \begin{cases} 0, & j = 1 \\[6pt] \text{next}[j], & T[j] \neq T[\text{next}[j]] \\[6pt] \text{nextval}[\text{next}[j]], & T[j] = T[\text{next}[j]] \end{cases}$$

用人话说：如果跳过去的位置字符和当前一样，就继续跳，直到字符不同或到达 0。

### 四、代码实现

**方法一：先求 next，再由 next 推 nextval**

In [ ]:
void GetNextVal(SString T, int nextval[]) {
    int next[MAXLEN + 1];
    GetNext(T, next);      // 先计算 next 数组

    nextval[1] = 0;        // 固定值
    for (int j = 2; j <= T.length; j++) {
        if (T.ch[j] == T.ch[next[j]]) {
            nextval[j] = nextval[next[j]];  // 继续跳
        } else {
            nextval[j] = next[j];           // 不需要跳
        }
    }
}

**方法二：直接一步到位计算 nextval**（在求 next 的过程中顺便优化）

In [ ]:
void GetNextVal_Direct(SString T, int nextval[]) {
    int j = 1, k = 0;
    nextval[1] = 0;

    while (j < T.length) {
        if (k == 0 || T.ch[j] == T.ch[k]) {
            j++;
            k++;
            // 关键修改：判断 T[j] 和 T[k] 是否相等
            if (T.ch[j] != T.ch[k]) {
                nextval[j] = k;           // 不等，正常赋值
            } else {
                nextval[j] = nextval[k];  // 相等，继续沿用
            }
        } else {
            k = nextval[k];
        }
    }
}

### 五、完整示例

以 `T = "ababaaab"` 为例：

已知 next 数组：

| j | 1 | 2 | 3 | 4 | 5 | 6 | 7 | 8 |
|---|---|---|---|---|---|---|---|---|
| T[j] | a | b | a | b | a | a | a | b |
| next[j] | 0 | 1 | 1 | 2 | 3 | 4 | 2 | 2 |

逐个计算 nextval：

```
j=1: nextval[1] = 0（固定值）

j=2: T[2]='b', T[next[2]]= T[1]='a'
     'b' ≠ 'a' → nextval[2] = next[2] = 1

j=3: T[3]='a', T[next[3]]= T[1]='a'
     'a' == 'a' → nextval[3] = nextval[1] = 0

j=4: T[4]='b', T[next[4]]= T[2]='b'
     'b' == 'b' → nextval[4] = nextval[2] = 1

j=5: T[5]='a', T[next[5]]= T[3]='a'
     'a' == 'a' → nextval[5] = nextval[3] = 0

j=6: T[6]='a', T[next[6]]= T[4]='b'
     'a' ≠ 'b' → nextval[6] = next[6] = 4

j=7: T[7]='a', T[next[7]]= T[2]='b'
     'a' ≠ 'b' → nextval[7] = next[7] = 2

j=8: T[8]='b', T[next[8]]= T[2]='b'
     'b' == 'b' → nextval[8] = nextval[2] = 1
```

| j | 1 | 2 | 3 | 4 | 5 | 6 | 7 | 8 |
|---|---|---|---|---|---|---|---|---|
| T[j] | a | b | a | b | a | a | a | b |
| next[j] | 0 | 1 | 1 | 2 | 3 | 4 | 2 | 2 |
| nextval[j] | 0 | 1 | 0 | 1 | 0 | 4 | 2 | 1 |

### 六、nextval 对 KMP 的提升

使用 nextval 代替 next 后，KMP 的匹配代码完全不变，只是把 `next[]` 换成 `nextval[]`：

In [ ]:
int Index_KMP_Optimized(SString S, SString T, int nextval[]) {
    int i = 1, j = 1;
    while (i <= S.length && j <= T.length) {
        if (j == 0 || S.ch[i] == T.ch[j]) {
            i++;
            j++;
        } else {
            j = nextval[j];  // 用 nextval 替换 next
        }
    }
    if (j > T.length)
        return i - T.length;
    else
        return 0;
}

时间复杂度不变（仍然是 O(n+m)），但**常数因子更小**，减少了不必要的比较。

### 七、next vs nextval 总结

| 特性 | next | nextval |
|------|------|---------|
| 信息来源 | 最长相等真前后缀 | 在 next 基础上消除冗余跳转 |
| 适用场景 | 通用 | 模式串有大量重复字符时效果显著 |
| 考试要求 | **必须掌握** | **必须掌握** |
| 工程实际 | 经典 KMP 实现常用 | 优化版本 |

---

## 第三部分：特殊矩阵的压缩存储

---

## 3.4.1 - 3.4.4 特殊矩阵的压缩存储

### 核心问题

对于 n×n 的矩阵，普通二维数组需要 n² 个存储空间。但很多矩阵具有特殊结构（大量重复元素或零元素），可以利用这些规律进行**压缩存储**，节省空间。

### 一、对称矩阵

#### 定义

若 n 阶方阵满足 `a[i][j] = a[j][i]`（对任意 i, j），则称为**对称矩阵**。

```
     1  5  1  3
     5  0  8  0
     1  8  9  2
     3  0  2  6
```

上三角和下三角完全对称，只需存储一半。

#### 压缩方案

只存储**下三角区 + 主对角线**（即 `i ≥ j` 的元素），共 `n(n+1)/2` 个元素，存入一维数组 `B[0..n(n+1)/2-1]`。

**存储顺序**：按行优先（先存第 1 行的下三角部分，再存第 2 行的...）

```
存储顺序：
a₁₁
a₂₁ a₂₂
a₃₁ a₃₂ a₃₃
a₄₁ a₄₂ a₄₃ a₄₄

B = [a₁₁, a₂₁, a₂₂, a₃₁, a₃₂, a₃₃, a₄₁, a₄₂, a₄₃, a₄₄]
下标:  0    1    2    3    4    5    6    7    8    9
```

#### 地址映射公式

矩阵元素 `a[i][j]`（**行列均从 1 开始**）在一维数组 B 中的下标 k：

$$k = \begin{cases} \dfrac{i(i-1)}{2} + j - 1, & i \geq j \text{（下三角区和对角线）} \\[10pt] \dfrac{j(j-1)}{2} + i - 1, & i < j \text{（上三角区，利用对称性 } a_{ij}=a_{ji}\text{）} \end{cases}$$

**推导过程**（以 `i ≥ j` 为例）：

```
第 1 行存了 1 个元素
第 2 行存了 2 个元素
...
第 i-1 行存了 i-1 个元素
共 1+2+...+(i-1) = i(i-1)/2 个元素

a[i][j] 是第 i 行的第 j 个元素

所以 k = i(i-1)/2 + j - 1（下标从 0 开始）
```

#### 代码实现

In [ ]:
#define N 4

In [ ]:
int B[N * (N + 1) / 2];

In [ ]:
// 压缩数组

// 获取对称矩阵元素 a[i][j]，i,j 从 1 开始
int getElement(int B[], int i, int j) {
    if (i >= j)
        return B[i * (i - 1) / 2 + j - 1];
    else
        return B[j * (j - 1) / 2 + i - 1];  // 利用对称性
}

// 设置对称矩阵元素

In [ ]:
void setElement(int B[], int i, int j, int val) {
    if (i >= j)
        B[i * (i - 1) / 2 + j - 1] = val;
    else
        B[j * (j - 1) / 2 + i - 1] = val;
}

### 二、三角矩阵

#### 定义

- **下三角矩阵**：上三角区（不含对角线）的元素全部相同（通常为常数 c）
- **上三角矩阵**：下三角区（不含对角线）的元素全部相同

```
下三角矩阵：             上三角矩阵：
1  c  c  c               1  5  1  3
5  2  c  c               c  2  8  7
1  8  3  c               c  c  3  2
3  6  2  4               c  c  c  4
```

#### 压缩方案（以下三角矩阵为例）

存储**下三角区 + 对角线** 的 `n(n+1)/2` 个元素，再加上 1 个位置存储常数 c。

共需 `n(n+1)/2 + 1` 个存储空间。

#### 地址映射公式

$$k = \begin{cases} \dfrac{i(i-1)}{2} + j - 1, & i \geq j \text{（下三角区和对角线）} \\[10pt] \dfrac{n(n+1)}{2}, & i < j \text{（上三角区，都映射到存放常数 c 的位置）} \end{cases}$$

#### 上三角矩阵的映射公式

对于上三角矩阵，存储上三角区 + 对角线：

$$k = \begin{cases} \dfrac{(2n - i + 2)(i - 1)}{2} + (j - i), & i \leq j \text{（上三角区和对角线）} \\[10pt] \dfrac{n(n+1)}{2}, & i > j \text{（下三角区，映射到常数 c 的位置）} \end{cases}$$

**推导**（上三角，行优先）：

```
第 1 行存了 n 个元素
第 2 行存了 n-1 个元素
...
第 i-1 行存了 n-i+2 个元素

前 i-1 行共存了：n + (n-1) + ... + (n-i+2) = (2n-i+2)(i-1)/2 个元素

a[i][j] 在第 i 行是第 (j-i+1) 个元素（从第 i 列开始）

所以 k = (2n-i+2)(i-1)/2 + (j-i)
```

### 三、三对角矩阵（带状矩阵）

#### 定义

n 阶方阵中，只有**主对角线**及其**相邻两条对角线**上的元素非零，其余全为 0。

也就是说：当 `|i - j| > 1` 时，`a[i][j] = 0`。

```
a₁₁ a₁₂  0   0   0
a₂₁ a₂₂ a₂₃  0   0
 0  a₃₂ a₃₃ a₃₄  0
 0   0  a₄₃ a₄₄ a₄₅
 0   0   0  a₅₄ a₅₅
```

#### 非零元素个数

- 第 1 行：2 个（`a₁₁, a₁₂`）
- 第 2 到 n-1 行：每行 3 个
- 第 n 行：2 个（`aₙ,ₙ₋₁, aₙₙ`）
- 总计：`2 + 3(n-2) + 2 = 3n - 2` 个

#### 地址映射公式

将非零元素按行优先存入一维数组 `B[0..3n-3]`。

已知 `a[i][j]` 为非零元素（即 `|i-j| ≤ 1`），求其在 B 中的下标 k：

$$k = (3i - 4) + (j - i + 2) - 1 = 2i + j - 3$$

**推导**：

```
第 1 行前有 0 个非零元素
第 2 行前有 2 个非零元素
第 3 行前有 2+3=5 个非零元素
...
第 i 行前有 2 + 3(i-2) = 3i-4 个非零元素 （i≥2）

a[i][j] 在第 i 行中：
  如果 j=i-1，是第 1 个
  如果 j=i，  是第 2 个
  如果 j=i+1，是第 3 个
  即第 (j-i+2) 个

k = (3i-4) + (j-i+2) - 1 = 2i + j - 3  （下标从 0 开始）

对于 i=1：k = 2+j-3 = j-1
  j=1 → k=0 ✓
  j=2 → k=1 ✓
```

#### 反推公式

已知下标 k，反推 `i, j`：

$$i = \lfloor(k + 1) / 3\rfloor + 1, \quad j = k - 2i + 3$$

该公式在 `k = 0` 时会得到第 1 行；其余下标也能对应到所属行的 2 或 3 个非零元素。

#### 代码实现

In [ ]:
// 三对角矩阵压缩存储

In [ ]:
int B[3 * N - 2];

In [ ]:
// N 阶三对角矩阵

// a[i][j] → B[k]
int triDiagGet(int B[], int n, int i, int j) {
    if (abs(i - j) > 1) return 0;  // 非带状区域
    int k = 2 * i + j - 3;
    return B[k];
}

// 设置元素

In [ ]:
void triDiagSet(int B[], int n, int i, int j, int val) {
    if (abs(i - j) > 1) return;  // 非带状区域不可设置
    int k = 2 * i + j - 3;
    B[k] = val;
}

### 四、稀疏矩阵

#### 定义

矩阵中非零元素的个数 t 远小于总元素个数 m×n，即**非零元素占比非常小**。

通常当非零元素占比低于 5% 时就认为是稀疏矩阵。

```
0  0  0  0  5
0  0  1  0  0
0  0  0  0  0
2  0  0  0  0
0  7  0  0  0

5×5 矩阵只有 4 个非零元素，非零率 = 4/25 = 16%（这里仅用于说明三元组存储）
```

#### 方法一：三元组表（顺序存储）

用三元组 `(行号, 列号, 值)` 存储每一个非零元素。

In [ ]:
#define MAXSIZE 100  // 非零元素最大个数

In [ ]:
typedef struct {
    int row, col;  // 行号、列号（从 1 开始）
    int val;       // 元素值
} Triple;
typedef struct {
    Triple data[MAXSIZE + 1]; // data[0] 不用
    int rows, cols, nums;     // 矩阵行数、列数、非零元素个数
} TSMatrix;

上面的稀疏矩阵存储为：

| 编号 | row | col | val |
|------|-----|-----|-----|
| 1 | 1 | 5 | 5 |
| 2 | 2 | 3 | 1 |
| 3 | 4 | 1 | 2 |
| 4 | 5 | 2 | 7 |

`rows=5, cols=5, nums=4`

**优点**：节省空间（只存非零元素）
**缺点**：不能随机访问，查找某个元素需要遍历三元组表

#### 方法二：十字链表（链式存储）

每个非零元素用一个结点表示，同时挂在**行链表**和**列链表**上。

In [ ]:
typedef struct OLNode {
    int row, col;           // 行号、列号
    int val;                // 元素值
    struct OLNode *right;   // 同一行的下一个非零元素
    struct OLNode *down;    // 同一列的下一个非零元素
} OLNode;
typedef struct {
    OLNode **rhead;  // 行链表头指针数组
    OLNode **chead;  // 列链表头指针数组
    int rows, cols, nums;
} CrossList;

结构示意：

```
         col1    col2    col3    col4    col5
row1: ──→ ─── ──→ ─── ──→ ─── ──→ ─── ──→[1,5,5]
          ↓                               ↓
row2: ──→ ─── ──→ ─── ──→[2,3,1]──→ ─── ──→ ───
          ↓              ↓
row3: ──→ ─── ──→ ─── ──→ ─── ──→ ─── ──→ ───
          ↓
row4: ──→[4,1,2]──→ ─── ──→ ─── ──→ ─── ──→ ───
          ↓
row5: ──→ ─── ──→[5,2,7]──→ ─── ──→ ─── ──→ ───
```

每个非零元素结点有两个指针：`right` 指向同行的下一个非零元素，`down` 指向同列的下一个非零元素。

**优点**：插入/删除非零元素方便，可以高效地按行或按列遍历
**缺点**：存储开销大（每个结点需要额外的两个指针）

### 五、压缩存储方法总结

| 矩阵类型 | 压缩方法 | 空间复杂度 | 随机访问 |
|---------|---------|-----------|---------|
| 对称矩阵 | 只存下三角 | n(n+1)/2 | ✅ O(1) |
| 三角矩阵 | 只存三角部分 + 1个常数 | n(n+1)/2 + 1 | ✅ O(1) |
| 三对角矩阵 | 只存带状区域 | 3n - 2 | ✅ O(1) |
| 稀疏矩阵 | 三元组表 / 十字链表 | O(t) | ❌ / ❌ |

> 📌 **考试重点**：对称矩阵和三角矩阵的映射公式是**选择题/填空题高频考点**，务必熟记。三对角矩阵公式也需掌握。稀疏矩阵理解概念和两种存储方式即可。

---

## 全章总结与核心考点

### 串部分

| 知识点 | 考试要求 | 重要程度 |
|--------|---------|---------|
| 串的定义与术语 | 理解概念 | ⭐⭐ |
| 三种存储结构 | 理解异同 | ⭐⭐ |
| 朴素模式匹配 | 理解思想，能写代码 | ⭐⭐⭐ |
| KMP 匹配算法 | 理解思想，能写代码 | ⭐⭐⭐⭐⭐ |
| 手算 next 数组 | **必考**，选择题/填空题 | ⭐⭐⭐⭐⭐ |
| 代码求 next 数组 | 理解递推过程 | ⭐⭐⭐⭐ |
| nextval 数组 | 手算 + 理解优化原理 | ⭐⭐⭐⭐ |

### 矩阵压缩部分

| 知识点 | 考试要求 | 重要程度 |
|--------|---------|---------|
| 对称矩阵压缩 | **映射公式必考** | ⭐⭐⭐⭐⭐ |
| 三角矩阵压缩 | 映射公式 | ⭐⭐⭐⭐ |
| 三对角矩阵压缩 | 映射公式 | ⭐⭐⭐ |
| 稀疏矩阵 | 三元组表和十字链表概念 | ⭐⭐⭐ |

### 手算 next/nextval 速查口诀

1. `next[1] = 0`（固定）
2. `next[2] = 1`（固定，因为长度为 1 的串没有真前后缀）
3. 从 j=3 开始，看 `T[1..j-1]` 的最长相等真前后缀长度 L，则 `next[j] = L + 1`
4. nextval：在 next 基础上，若 `T[j] == T[next[j]]`，则 `nextval[j] = nextval[next[j]]`

> 🏆 **终极建议**：KMP 的 next 数组手算一定要多练。考试中通常给一个模式串让你写出 next 和 nextval 数组，分值在 4-8 分，务必拿满分。

---

以上就是**串与压缩存储**的完整教程。建议学习顺序：先理解朴素匹配 → 再啃 KMP → 反复手算 next/nextval → 最后把矩阵压缩公式背熟。祝学习顺利！🎯